In [14]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.tensorboard.writer import SummaryWriter

import trimesh

import network, utils

import pathlib
import os
import shutil
from collections import defaultdict
import datetime



%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:

if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
else:
    print("Using CPU")
    device = torch.device("cpu")

pcd = trimesh.load(pathlib.Path("data/pointcloud10.obj"), file_type = 'obj', force='pointcloud')
target_pc = torch.tensor(pcd.vertices, dtype=torch.float32).to(device)
target_pc = target_pc.unsqueeze(0) # Change [2048, 3] -> [1, 2048, 3]
# print(target_pc) # check

# 2. Initialize Model and Diffuser
denoiser = network.Denoiser().to(device)
diffuser = network.Diffuser(timesteps=1000).to(device)
optimizer = Adam(denoiser.parameters(), lr=1e-4)

# Model name
current_time = datetime.datetime.now().strftime("%b%d_%H-%M-%S")
experiment_name = "10000_epochs"
denoiser_id = f"{experiment_name}_{current_time}"

# Create tensorboard writer    
log_path = pathlib.Path(f"logs/diffusion_training/{denoiser_id}")
writer = SummaryWriter(log_path)
# For tracking loss per timestep
timestep_loss_sum = defaultdict(float)
timestep_counts = defaultdict(int)
#Run this code in terminal to start tensorboard: tensorboard --logdir=diffusion/nikola/logs/diffusion_training


# 3. Training Loop
epochs = 10000 
for epoch in range(epochs):
    denoiser.train()
    optimizer.zero_grad()
    
    # Pick a random timestep for each item in the batch
    # Since batch size is 1, we just pick one t
    t = torch.randint(0, diffuser.t, (1,), device=device).long()
    
    # Forward Process: Add noise to the clean shape
    noisy_pc, actual_noise = diffuser.add_noise(target_pc, t)
    
    # Backward Process: Predict the noise
    predicted_noise = denoiser(noisy_pc, t)
    
    # Loss: How well did the model guess the noise?
    loss = F.mse_loss(predicted_noise, actual_noise)
    
    loss.backward()
    optimizer.step()

    writer.add_scalar("Loss", loss.item(), epoch)
    timestep_loss_sum[t.item()] += loss.item()
    timestep_counts[t.item()] += 1
    print(f"Epoch {epoch}, T = {t.item()} | Loss: {loss.item():.6f}")

for time_val in range(diffuser.t):
        if timestep_counts[time_val] > 0:
            avg_error = timestep_loss_sum[time_val] / timestep_counts[time_val]
            writer.add_scalar('Timestep Error', avg_error, global_step=time_val)


utils.save_model(denoiser, diffuser, epoch, denoiser_id)


Using GPU: NVIDIA GeForce MX130
Epoch 0, T = 791 | Loss: 1.043209
Epoch 1, T = 499 | Loss: 0.962604
Epoch 2, T = 566 | Loss: 0.890117
Epoch 3, T = 350 | Loss: 0.852268
Epoch 4, T = 930 | Loss: 0.799853
Epoch 5, T = 848 | Loss: 0.752943
Epoch 6, T = 594 | Loss: 0.737255
Epoch 7, T = 59 | Loss: 0.906877
Epoch 8, T = 166 | Loss: 0.754176
Epoch 9, T = 311 | Loss: 0.656431
Epoch 10, T = 228 | Loss: 0.640321
Epoch 11, T = 979 | Loss: 0.497615
Epoch 12, T = 20 | Loss: 0.994421
Epoch 13, T = 791 | Loss: 0.421176
Epoch 14, T = 221 | Loss: 0.523282
Epoch 15, T = 539 | Loss: 0.372836
Epoch 16, T = 12 | Loss: 1.047495
Epoch 17, T = 176 | Loss: 0.507991
Epoch 18, T = 935 | Loss: 0.247947
Epoch 19, T = 372 | Loss: 0.268257
Epoch 20, T = 934 | Loss: 0.179470
Epoch 21, T = 186 | Loss: 0.418953
Epoch 22, T = 139 | Loss: 0.515969
Epoch 23, T = 974 | Loss: 0.113152
Epoch 24, T = 832 | Loss: 0.103695
Epoch 25, T = 237 | Loss: 0.282681
Epoch 26, T = 609 | Loss: 0.071835
Epoch 27, T = 723 | Loss: 0.066186
E

In [ ]:
#Load a model:
denoiser_id = ""
utils.reload_model(denoiser, diffuser, denoiser_id, device)

In [34]:
generateDDIM = True
generateDDPM = True
number_of_points = 1024
DDIM_steps = 30

if generateDDIM:
    generated_pc_ddim = network.sample_ddim(denoiser, diffuser, n_points=number_of_points, steps=DDIM_steps)
    generated_pcd_ddim = trimesh.PointCloud(generated_pc_ddim.squeeze().cpu().numpy())
    utils.visualize_comparison(target_pc, generated_pc_ddim, window_name="DDIM Target (Red) vs Generated (Blue)")

if generateDDPM:
    generated_pc_ddpm = network.sample_ddpm(denoiser, diffuser, n_points=number_of_points)
    generated_pcd_ddpm = trimesh.PointCloud(generated_pc_ddpm.squeeze().cpu().numpy())
    utils.visualize_comparison(target_pc, generated_pc_ddpm, window_name="DDPM Target (Red) vs Generated (Blue)")




Visualizing: Target is RED, Generated is BLUE.
Visualizing: Target is RED, Generated is BLUE.


In [7]:
generated_pc, samples_list = network.sample_and_capture(denoiser, diffuser, n_points=number_of_points, save_every=10)
utils.visualize_diffusion_progress(samples_list, window_name="Diffusion Process")

In [35]:

# Optionally, save the generated point clouds to disk
generated_pcd_ddim.export(f"output/{denoiser_id}_ddim.obj")
generated_pcd_ddpm.export(f"output/{denoiser_id}_ddpm.obj")

'# https://github.com/mikedh/trimesh\nv -0.56441766 0.78872430 -0.18467656\nv 0.69580579 -0.20013197 0.61616433\nv -0.86288488 0.49203083 -0.15962587\nv 0.48783290 0.35668495 0.40350926\nv 0.98029727 0.09194494 -0.21980199\nv 0.02946977 -0.98523468 -0.26776981\nv 0.00129082 0.86175340 0.01463670\nv 0.53662318 -0.27286988 0.50937229\nv -0.26935858 0.32102132 -0.18292204\nv -0.33940518 0.38525423 -0.46772867\nv -0.69208896 -0.21610272 0.18053158\nv -0.68740106 -0.43805766 -0.26573423\nv -0.52584171 -0.30063847 0.30289534\nv 0.05930847 -0.19985540 -0.29118839\nv -0.02350364 0.86983013 -0.38706601\nv 0.81569612 0.53392655 -0.34571153\nv -1.06174374 -0.03352182 0.02351772\nv 0.07378376 -0.77049774 -0.10110997\nv 0.25538582 -0.28099233 0.58130467\nv -0.25627917 0.23873901 -0.42154562\nv -0.65982777 0.55066711 -0.30719045\nv -0.80864823 -0.60709095 0.05355192\nv -0.57824832 0.52951932 -0.42947111\nv 1.06332219 -0.11305494 -0.09279465\nv 0.27940497 0.43306753 -0.00727826\nv -0.79350215 -0.4030